# 阶段四：超时边界

使用 `asyncio.timeout()` 为单个工具、工具批次和整次请求设置超时。

In [ ]:
import asyncio

async def fake_tool(name: str, delay: float) -> str:
    """模拟外部 I/O。

    参数:
        name: 工具名称。
        delay: 模拟等待秒数。

    返回:
        工具完成后的结果。
    """
    try:
        await asyncio.sleep(delay)
        return f"{name}: done"
    finally:
        print(f"{name}: cleanup")

async def run_with_timeout(name: str, delay: float, timeout: float) -> str:
    """执行单个工具并处理超时。

    参数:
        name: 工具名称。
        delay: 工具运行时间。
        timeout: 最大运行秒数。

    返回:
        成功结果或超时结果。
    """
    try:
        async with asyncio.timeout(timeout):
            return await fake_tool(name, delay)
    except TimeoutError:
        return f"{name}: timeout"

async def run_batch(timeout: float) -> list[str]:
    """并发执行工具，并为每个工具设置独立超时。

    参数:
        timeout: 每个工具的最大运行秒数。

    返回:
        按提交顺序排列的结果列表。
    """
    return await asyncio.gather(
        run_with_timeout("fast_tool", 0.05, timeout),
        run_with_timeout("slow_tool", 0.2, timeout),
    )

async def run_request(tool_timeout: float, batch_timeout: float) -> list[str]:
    """为整个工具批次增加总超时。

    参数:
        tool_timeout: 单个工具的最大运行秒数。
        batch_timeout: 整个批次的最大运行秒数。

    返回:
        批次结果；整批超时时返回空列表。
    """
    try:
        async with asyncio.timeout(batch_timeout):
            return await run_batch(tool_timeout)
    except TimeoutError:
        return []

async def main() -> None:
    """验证单工具超时、整批超时和清理行为。"""
    results = await run_request(tool_timeout=0.1, batch_timeout=1.0)
    print(results)
    assert results == ["fast_tool: done", "slow_tool: timeout"]

    assert await run_request(tool_timeout=1.0, batch_timeout=0.05) == []
    print("all timeout checks passed")

await main()


In [ ]:
from langchain_core.runnables import RunnableLambda

async def langchain_async_example() -> None:
    """为 LangChain Runnable 的 `ainvoke` 增加超时边界。"""
    runnable = RunnableLambda(afunc=lambda name: fake_tool(name, 0.2))
    try:
        async with asyncio.timeout(0.05):
            await runnable.ainvoke("slow_tool")
    except TimeoutError:
        print("LangChain Async: timeout")
    else:
        raise AssertionError("expected timeout")

await langchain_async_example()
